# 🛠️ 03_processed_validation_eda: Pipeline Sanity Check
This notebook verifies the output of the `feature_pipeline.py`. We ensure that transformations like cyclical encoding and scaling were applied correctly before the data hits the model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pathlib

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load final processed data
X = pd.read_parquet("../data/processed/X_train.parquet")
y = pd.read_parquet("../data/processed/y_train.parquet")
df = X.copy()
df['target'] = y['target']

print(f"Processed data loaded. Shape: {df.shape}")
df.head()

## 1. Cyclical Encoding Verification
We plot `hour_sin` vs `hour_cos`. If the logic is correct, these should form a perfect unit circle.

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(df['hour_sin'], df['hour_cos'], s=1, alpha=0.2)
plt.title("Cyclical Encoding Check: Hour Sin vs Cos")
plt.xlabel("Hour Sin")
plt.ylabel("Hour Cos")
plt.axis('equal')
plt.show()

## 2. Scaling Verification
We check the distribution of numerical features after the `RobustScaler`. They should be centered around 0.

In [ ]:
# Identify numerical columns (scaled ones)
num_cols = ['passenger_count', 'hour_sin', 'hour_cos', 'day_of_week', 'haversine_dist_km']
# Filter only columns that exist in df to avoid errors
existing_num_cols = [c for c in num_cols if c in df.columns]

df[existing_num_cols].hist(bins=30, figsize=(15, 10))
plt.suptitle("Distributions of Scaled Features")
plt.show()

### ✍️ Engineering Inferences:
1. **Cycle Check:** The perfect circle in the sin/cos plot confirms the temporal encoding is mathematically sound.
2. **Scaling Check:** The `RobustScaler` handled the outliers well, keeping the bulk of the data centered without squashing it.
3. **Ready for Training:** Data is scaled, cleaned, and logically sound. We are ready for the model.